# Extract & Label Chess Piece Glyphs from PDF

This notebook helps you manually extract and label piece glyphs from a chess PDF book.

**Output:** A folder structure of labeled glyphs ready for training:
```
glyphs/
├── K/  (King glyphs)
├── Q/  (Queen glyphs)
├── R/  (Rook glyphs)
├── B/  (Bishop glyphs)
└── N/  (Knight glyphs)
```

## Step 1 — Mount Google Drive and load PDF

In [ ]:
from google.colab import drive
import os

drive.mount('/content/gdrive')

In [ ]:
# ── EDIT THIS: set your PDF path ────────────────────────────────────────────
PDF_PATH = '/content/gdrive/MyDrive/chess_book.pdf'  # CHANGE THIS to your PDF

# Verify PDF exists
if not os.path.exists(PDF_PATH):
    print(f'❌ PDF not found: {PDF_PATH}')
    print(f'   Available items in /content/gdrive/MyDrive:')
    for item in os.listdir('/content/gdrive/MyDrive')[:20]:
        print(f'     - {item}')
else:
    size_mb = os.path.getsize(PDF_PATH) / (1024*1024)
    print(f'✅ PDF loaded: {PDF_PATH}  ({size_mb:.1f} MB)')

## Step 2 — Install dependencies

In [ ]:
!apt-get install -y poppler-utils
!pip install -q pdfplumber pdf2image pillow

## Step 3 — Configuration

In [ ]:
import pdfplumber
from pdf2image import convert_from_path
from PIL import Image
import os
from pathlib import Path
import shutil

# Configuration
START_PAGE = 1
END_PAGE = 20
GLYPH_DPI = 150
IMG_SIZE = 32

# Output directory
GLYPHS_DIR = './glyphs'
PIECE_CLASSES = ['K', 'Q', 'R', 'B', 'N']

# Create output directories
Path(GLYPHS_DIR).mkdir(exist_ok=True)
for piece in PIECE_CLASSES:
    Path(f'{GLYPHS_DIR}/{piece}').mkdir(exist_ok=True)

print(f'✅ Output directory: {GLYPHS_DIR}/')
print(f'   Subdirectories for: {" ".join(PIECE_CLASSES)}')

## Step 4 — Find words with glyphs in PDF

In [ ]:
def has_non_ascii_or_special(text):
    """Check if text contains non-ASCII/special chars (likely a glyph)."""
    for char in text:
        if char in '♔♕♖♗♘♙♚♛♜♝♞♟':
            return True
        if ord(char) > 127:
            return True
        if char in '0123456789)£@~`':
            return True
    return False

def render_page(pdf_path, page_num, dpi=150):
    images = convert_from_path(pdf_path, first_page=page_num+1, last_page=page_num+1, dpi=dpi)
    return images[0] if images else None

# Find all words with glyphs
glyph_words = []

with pdfplumber.open(PDF_PATH) as pdf:
    pdf_page_count = len(pdf.pages)
    if END_PAGE > pdf_page_count:
        END_PAGE = pdf_page_count
    
    for page_idx in range(START_PAGE - 1, END_PAGE):
        page_num = page_idx + 1
        pdf_page = pdf.pages[page_idx]
        
        try:
            words = pdf_page.extract_words()
        except:
            continue
        
        for word in words:
            text = word.get('text', '')
            
            # Find words with glyphs
            if has_non_ascii_or_special(text):
                glyph_words.append({
                    'page': page_num,
                    'text': text,
                    'bbox': (word['x0'], word['top'], word['x1'], word['bottom']),
                })

print(f'✅ Found {len(glyph_words)} words with glyphs')
print(f'\nFirst 20 examples:')
for i, w in enumerate(glyph_words[:20], 1):
    print(f'  {i:2}. P{w["page"]}: "{w["text"]}"')

## Step 5 — Manual labeling UI

For each word with glyphs, you'll:
1. See the image of the word from the PDF
2. Select which piece it is: K, Q, R, B, N, or Skip
3. The glyph gets saved to the corresponding folder

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Image as IPImage, clear_output
import io
from PIL import Image as PILImage

# Render all pages once (cache)
rendered_pages = {}

def get_rendered_page(page_num):
    if page_num not in rendered_pages:
        rendered_pages[page_num] = render_page(PDF_PATH, page_num - 1, dpi=GLYPH_DPI)
    return rendered_pages[page_num]

# UI State
current_idx = 0
saved_count = {'K': 0, 'Q': 0, 'R': 0, 'B': 0, 'N': 0}
skipped = 0
discarded = 0

def show_glyph(idx):
    """Display a glyph and show labeling buttons."""
    global current_idx, saved_count, skipped, discarded
    
    if idx >= len(glyph_words):
        clear_output()
        print(f'✅ Labeling complete!')
        print(f'\nSaved:')
        for piece in PIECE_CLASSES:
            print(f'  {piece}: {saved_count[piece]}')
        print(f'  Skipped: {skipped}')
        print(f'  Discarded: {discarded}')
        return
    
    current_idx = idx
    word_info = glyph_words[idx]
    page_num = word_info['page']
    bbox = word_info['bbox']
    
    # Crop glyph image
    page_image = get_rendered_page(page_num)
    if page_image is None:
        show_glyph(idx + 1)
        return
    
    scale = GLYPH_DPI / 72.0
    x0 = max(0, int(bbox[0] * scale))
    y0 = max(0, int(bbox[1] * scale))
    x1 = min(page_image.width, int(bbox[2] * scale))
    y1 = min(page_image.height, int(bbox[3] * scale))
    
    crop = page_image.crop((x0, y0, x1, y1))
    
    # Display
    clear_output()
    print(f'Glyph {idx + 1}/{len(glyph_words)} — P{page_num}: "{word_info["text"]}"')
    print()
    display(crop)
    print()
    
    # Buttons
    def save_glyph(piece):
        # Save to piece folder
        count = saved_count[piece]
        filename = f'{GLYPHS_DIR}/{piece}/{count + 1:04d}.png'
        crop.save(filename)
        saved_count[piece] += 1
        show_glyph(idx + 1)
    
    def skip():
        global skipped
        skipped += 1
        show_glyph(idx + 1)
    
    def discard():
        global discarded
        discarded += 1
        show_glyph(idx + 1)
    
    buttons = [
        widgets.Button(description='K (King)', button_style='info'),
        widgets.Button(description='Q (Queen)', button_style='info'),
        widgets.Button(description='R (Rook)', button_style='info'),
        widgets.Button(description='B (Bishop)', button_style='info'),
        widgets.Button(description='N (Knight)', button_style='info'),
        widgets.Button(description='Skip', button_style='warning'),
        widgets.Button(description='❌ Discard', button_style='danger'),
    ]
    
    buttons[0].on_click(lambda _: save_glyph('K'))
    buttons[1].on_click(lambda _: save_glyph('Q'))
    buttons[2].on_click(lambda _: save_glyph('R'))
    buttons[3].on_click(lambda _: save_glyph('B'))
    buttons[4].on_click(lambda _: save_glyph('N'))
    buttons[5].on_click(lambda _: skip())
    buttons[6].on_click(lambda _: discard())
    
    display(widgets.HBox(buttons))

# Start
show_glyph(0)

## Step 6 — Summary & Export

In [ ]:
# Count saved glyphs
import os
from collections import defaultdict

counts = defaultdict(int)
for piece in PIECE_CLASSES:
    path = f'{GLYPHS_DIR}/{piece}'
    if os.path.exists(path):
        counts[piece] = len([f for f in os.listdir(path) if f.endswith('.png')])

print('✅ Labeling Complete!')
print(f'\nGlyphs summary:')
total = 0
for piece in PIECE_CLASSES:
    count = counts[piece]
    print(f'  {piece}: {count}')
    total += count

print(f'  ❌ Discarded: {discarded}')
print(f'\n  TOTAL SAVED: {total} glyphs')
print(f'\n📁 Folder structure ready for training:')
print(f'   glyphs/')
for piece in PIECE_CLASSES:
    count = counts[piece]
    print(f'   ├── {piece}/ ({count} images)')

print(f'\n💾 Download the glyphs/ folder and upload to training notebook')